File IDs work with ICA but I did not find a way to make it work in the pipepline.
So still need to get the file path :(

In [1]:
import pandas as pd
import glob
import csv

In [4]:
# Get file ids from existing summary to retrieve CRAM file paths
fn = '/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/ICA_cram_file_info/agd_cram_file_ids_from_2025_q1_release.tsv'
df_file_id = pd.read_csv(fn, sep='\t')

# There is one file with missing values
sample_id = 'R292665402'
cram_id = 'fil.cd547ff0db0d4120615908dd6e872de5'
cram_index_id = 'fil.9cec2b71025149969fbd08dd6eb04e50'
mask = df_file_id['sample_id']==sample_id
df_file_id.loc[mask, 'cram_id'] = cram_id
df_file_id.loc[mask, 'cram_index_id'] = cram_index_id

print(df_file_id.shape)
display(df_file_id.head())

output_fn = '/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/ICA_cram_file_info/agd_cram_file_ids_from_2025_q1_release.na_fixed.tsv'
df_file_id.to_csv(output_fn, index=False, sep='\t')

(250105, 3)


,sample_id,cram_id,cram_index_id
0,R284640820,fil.e1d940195cfb4506496a08dd5cea6966,fil.9114d2042c6142a3e9c808dd5cea44db
1,R228770742,fil.f97ade6fe66943ea859e08dd5cc3e5ec,fil.5427fd08d0b04b3f96eb08dd5cd71faa
2,R204986377,fil.85dd9242b4ff42e9f81208dd67f0c550,fil.29d37de162e840433b5808dd66b80cd6
3,R289309189,fil.d83e1db81c8248f5fd6908dd5cd3feca,fil.f7ebce1d397848c97cd208dd5cc3e5ec
4,R218805938,fil.f2c2c6ef229c4cb1aba108dd6e46a4eb,fil.db730dc71c71491179d408dd6e2e43d0


```
icav2 projectdata get fil.e1d940195cfb4506496a08dd5cea6966 --project-id 5b12d2b6-d3fc-4c34-905e-28acd9a42926
```

In [12]:
df_file_id = pd.read_csv('/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/ICA_cram_file_info/agd_cram_file_ids_from_2025_q1_release.na_fixed.tsv', sep='\t')
data_id = ''
icav2_path = '/data100t1/home/wanying/downloaded_tools/icav2/icav2'
output_path = '/DC3/AGD250k_CRAM_info_only/individual_info'
cmd_fn = './bash_slurm/get_cram_info_from_ica.sh'
fh = open(cmd_fn, 'w')
c = 0
for cmd in icav2_path + ' projectdata get ' + df_file_id['cram_id'] + f' --project-id 5b12d2b6-d3fc-4c34-905e-28acd9a42926 > {output_path}/' + df_file_id['sample_id'] + '.ica_info.txt':
    fh.write(cmd+'\n')
    c += 1
    if c%1000==0:
        print(f'\r# CMD written: {c}', end='', flush=True)
print(f'\r# CMD written: {c}')
print('# Done')
fh.close()


# CMD written: 250105
# Done


In [ ]:
# Process and combine individual files
# Run in terminal `get_file_path_from_ICA.py`
result_path = '/DC3/AGD250k_CRAM_info_only/individual_info'
lst_sample_id, lst_file_path = [], []
c = 0
for fn in glob.glob(f'{result_path}/*.txt'):
    lst_sample_id.append(fn.split('/')[-1].split('.ica_info.')[0])
    with open(fn) as fh:
        file_path = 'NA'
        for line in fh:
            if line.startswith('details.path'):
                file_path = line.split('details.path')[-1].strip()
                break
        lst_file_path.append(file_path)
    c += 1
    if c%50==0:
        print(f'\r# CMD written: {c}', end='', flush=True)
print(f'\r# Files processed: {c}')
print('# Done')


In [ ]:
# 2. Check if all samples have been looked up
# Check the file path loaded， find missing ones (likely due to server error)
fn_path = '/DC3/AGD250k_CRAM_info_only/agd250k_file_path_on_ica.txt'
df_path = pd.read_csv(fn_path, sep='\t')
df_path[df_path['file_path'].isna()]

df_need_redo = df_file_id[~df_file_id['sample_id'].isin(df_path.dropna()['sample_id'])]

# Redo failed or missing ones
data_id = ''
icav2_path = '/data100t1/home/wanying/downloaded_tools/icav2/icav2'
output_path = '/DC3/AGD250k_CRAM_info_only/individual_info'
cmd_fn = './bash_slurm/get_cram_info_from_ica.redo_failed.sh'
fh = open(cmd_fn, 'w')
c = 0
for cmd in icav2_path + ' projectdata get ' + df_need_redo['cram_id'] + f' --project-id 5b12d2b6-d3fc-4c34-905e-28acd9a42926 > {output_path}/' + df_need_redo['sample_id'] + '.ica_info.txt':
    fh.write(cmd+'\n')
    c += 1
    if c%1000==0:
        print(f'\r# CMD written: {c}', end='', flush=True)
print(f'\r# CMD written: {c}')
print('# Done')
fh.close()


# CMD written: 20909
# Done


In [7]:
# 3. Check if all samples have been looked up (again)
# Check the file path loaded， find missing ones (likely due to server error)
fn_path = '/DC3/AGD250k_CRAM_info_only/agd250k_file_path_on_ica.txt'
df_path = pd.read_csv(fn_path, sep='\t')
df_path[df_path['file_path'].isna()]

df_need_redo = df_file_id[~df_file_id['sample_id'].isin(df_path.dropna()['sample_id'])]

# Redo failed or missing ones
data_id = ''
icav2_path = '/data100t1/home/wanying/downloaded_tools/icav2/icav2'
output_path = '/DC3/AGD250k_CRAM_info_only/individual_info'
cmd_fn = './bash_slurm/get_cram_info_from_ica.redo_failed.redo.sh'
fh = open(cmd_fn, 'w')
c = 0
for cmd in icav2_path + ' projectdata get ' + df_need_redo['cram_id'] + f' --project-id 5b12d2b6-d3fc-4c34-905e-28acd9a42926 > {output_path}/' + df_need_redo['sample_id'] + '.ica_info.txt':
    fh.write(f'echo "# {c+1}"; ' + cmd+'\n')
    c += 1
    if c%1000==0:
        print(f'\r# CMD written: {c}', end='', flush=True)
print(f'\r# CMD written: {c}')
print('# Done')
fh.close()


# CMD written: 80
# Done


In [ ]:
# 3. Check if all samples have been looked up (again)
# Check the file path loaded， find missing ones (likely due to server error)
fn_path = '/DC3/AGD250k_CRAM_info_only/agd250k_file_path_on_ica.txt'
df_path = pd.read_csv(fn_path, sep='\t')
df_path[df_path['file_path'].isna()]

# Manually fixed these:
# sample_id, cram_id, cram_index_id
# R264171777, fil.0502556293ba4672bfa508dd65db89d2, fil.8b6681957f0744f0772108dd65dae063
# R222369458, fil.668c2197e91b4434edde08dd67f0c572, fil.e365589af369469ce4ac08dd67f0c572
# R258040009, fil.e924d4f81a95452ff66f08dd5b72798e, fil.e479d197355042d0af7e08dd5b727965


,sample_id,file_path


In [14]:
df = pd.read_csv('test.txt', sep=',')
output_path = '/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/ICA_cram_file_info'
df.to_csv(f'{output_path}/agd_cram_file_path_2_samples.no_quotes.tsv', index=False, sep='\t')

df = pd.read_csv('test.txt', sep=',')
df['cram_id'] = '"' + df['cram_id'] + '"'
df['cram_index_id'] = '"' + df['cram_index_id'] + '"'
output_path = '/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/ICA_cram_file_info'
df.to_csv(f'{output_path}/agd_cram_file_path_2_samples.double_quotes.tsv', quoting=csv.QUOTE_NONE, index=False, sep='\t')

df = pd.read_csv('test.txt', sep=',')
df['cram_id'] = "'" + df['cram_id'] + "'"
df['cram_index_id'] = "'" + df['cram_index_id'] + "'"
output_path = '/data100t1/home/wanying/BioVU/20260305_AGD_250k_HLA_typing/data/ICA_cram_file_info'
df.to_csv(f'{output_path}/agd_cram_file_path_2_samples.single_quotes.tsv', index=False, sep='\t')

255014

In [2]:
fn_path = '/DC3/AGD250k_CRAM_info_only/agd250k_file_path_on_ica.txt'
df_path = pd.read_csv(fn_path, sep='\t')
df_path[df_path['file_path'].isna()]

,sample_id,file_path
2561,R216647336,NaN
5738,R293953989,NaN
8520,R216812426,NaN
10356,R231641129,NaN
11381,R235876173,NaN
...,...,...
238535,R203344275,NaN
239092,R224564467,NaN
242165,R206517085,NaN
245161,R206274820,NaN
